# Submission Comparison Dashboard

This notebook provides an automated way to compare your latest submission against your best benchmark. Use this tool iteratively during model tuning to track performance shifts and verify consistency.

### 0. Generate Latest Submission (Optional)
Run this cell to update `submissions/submission.csv` using the current model and training pipeline. This ensures you are comparing the most up-to-date version.

In [ ]:
!python ../src/training/pipeline.py --mode prod --refit --output submission.csv

### 1. Configuration
Define the paths to the submissions you want to compare.

In [ ]:
CURRENT_SUB_PATH = '../submissions/submission.csv'
BEST_SUB_PATH = '../data/best_submit/best_750k.csv'

### 2. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Set plot styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

def load_and_align(curr_path, best_path):
    if not os.path.exists(curr_path):
        raise FileNotFoundError(f"Current submission not found: {curr_path}")
    if not os.path.exists(best_path):
        raise FileNotFoundError(f"Best submission not found: {best_path}")
        
    df_curr = pd.read_csv(curr_path, parse_dates=['Date'])
    df_best = pd.read_csv(best_path, parse_dates=['Date'])
    
    df = pd.merge(df_curr, df_best, on='Date', suffixes=('_curr', '_best'))
    return df

df = load_and_align(CURRENT_SUB_PATH, BEST_SUB_PATH)
df.head()

### 3. Interactive Comparison Dashboard
Compare the raw values of Revenue and COGS over time.

In [ ]:
def plot_comparison(df, target):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df['Date'], y=df[f'{target}_best'],
        name=f'{target} (Best)',
        line=dict(color='rgba(0, 0, 0, 0.4)', width=2, dash='dot')
    ))
    
    fig.add_trace(go.Scatter(
        x=df['Date'], y=df[f'{target}_curr'],
        name=f'{target} (Current)',
        line=dict(color='#636EFA', width=3)
    ))
    
    fig.update_layout(
        title=f'{target} Comparison: Current vs Best',
        xaxis_title='Date',
        yaxis_title=target,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig.show()

plot_comparison(df, 'Revenue')
plot_comparison(df, 'COGS')

### 4. Deviation & Bias Analysis
Identify where the current model deviates from the benchmark. A positive delta means the current forecast is higher than the best benchmark.

In [ ]:
for target in ['Revenue', 'COGS']:
    df[f'{target}_delta'] = df[f'{target}_curr'] - df[f'{target}_best']
    df[f'{target}_pct_diff'] = (df[f'{target}_delta'] / df[f'{target}_best']) * 100

def plot_delta(df, target):
    fig = go.Figure()
    
    # Area for delta
    fig.add_trace(go.Scatter(
        x=df['Date'], y=df[f'{target}_delta'],
        fill='tozeroy',
        name='Delta (Curr - Best)',
        line=dict(color='#EF553B')
    ))
    
    # Rolling average of delta
    fig.add_trace(go.Scatter(
        x=df['Date'], y=df[f'{target}_delta'].rolling(7).mean(),
        name='7-Day Rolling Mean of Delta',
        line=dict(color='#00CC96', width=2)
    ))
    
    fig.update_layout(
        title=f'{target} Deviation over Time',
        xaxis_title='Date',
        yaxis_title='Absolute Difference',
        hovermode='x unified',
        template='plotly_white'
    )
    fig.show()

plot_delta(df, 'Revenue')
plot_delta(df, 'COGS')

### 5. Distribution of Differences
Visualizing the scale and consistency of the changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, target in enumerate(['Revenue', 'COGS']):
    sns.histplot(df[f'{target}_pct_diff'], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'{target} % Difference Distribution')
    axes[i].set_xlabel('% Difference from Best')
    axes[i].axvline(0, color='red', linestyle='--')
    
plt.tight_layout()
plt.show()

### 6. Summary Statistics Dashboard

In [ ]:
stats = []
for target in ['Revenue', 'COGS']:
    curr_col = f'{target}_curr'
    best_col = f'{target}_best'
    
    mae = (df[curr_col] - df[best_col]).abs().mean()
    corr = df[curr_col].corr(df[best_col])
    mean_ratio = df[curr_col].mean() / df[best_col].mean()
    
    stats.append({
        'Target': target,
        'Pearson Corr': corr, # Keep as float for color styling
        'MAE vs Best': float(mae), # Keep as float to sum later
        'Scale Ratio (Curr/Best)': f"{mean_ratio:.4f}",
        'Abs % Diff Mean': f"{df[f'{target}_pct_diff'].abs().mean():.2f}%"
    })

# Total MAE calculation before string formatting
total_mae = sum([stat['MAE vs Best'] for stat in stats])

# Formatting the individual MAE values to strings for display
for stat in stats:
    stat['MAE vs Best'] = f"{stat['MAE vs Best']:,.2f}"

# Append summary row with np.nan for numeric columns
stats.append({
    'Target': 'TOTAL (Rev + COGS)',
    'Pearson Corr': np.nan, 
    'MAE vs Best': f"{total_mae:,.2f}",
    'Scale Ratio (Curr/Best)': '-',
    'Abs % Diff Mean': '-'
})

stats_df = pd.DataFrame(stats)

# Apply styling to float values, ignoring NaNs which automatically get skipped by Pandas Styler background_gradient
display(stats_df.style.set_caption("Summary Comparison Metrics").background_gradient(cmap='Blues', subset=['Pearson Corr']))